# CTA RouteRanker v3 — Colab training and model selection

This notebook follows the mentor's 10 Nov 2026 v3 guide: exact route/date lag features, a one-day reporting-delay assumption, seasonal baselines, pooled Linear Regression, Random Forest, and a retained HistGradientBoosting reference. Model choice is made using validation MAE; June–July test results are reported after selection.

**Data note:** the mentor's frozen ridership CSV was not provided. The notebook uses the current official Chicago Data Portal export (or a CSV you upload), records its SHA-256, and flags whether it matches the guide's frozen-file hash. A changed hash means this is an updated-source development run, not an exact reproduction. The June–July period was evaluated in earlier project iterations, so it is not a fresh final holdout.

Your eight GTFS archives were extracted automatically. Keep their folders under My Drive/CTA_GTFS, named with the full IDs such as mdb-389-202511230128. The notebook reads each folder in place; it does not need you to re-zip them.

In [ ]:
%pip -q install pandas numpy scikit-learn joblib
import hashlib, json, shutil, subprocess, sys, tempfile
from pathlib import Path

from google.colab import drive, files
drive.mount('/content/drive')

# Fresh checkout avoids stale source files in an earlier Colab runtime.
REPO = Path(tempfile.mkdtemp(prefix='route-ranker-v3-')) / 'repo'
subprocess.run([
    'git', 'clone', '--depth', '1', '--branch', 'users/bidhubhusan',
    'https://github.com/springboardmentor441p-coderr/Powered-Public-Transit-Route-Optimization.git',
    str(REPO),
], check=True)
sys.path.insert(0, str(REPO))
%cd $REPO
print('Project source:', REPO)

## 1. Get and fingerprint the ridership CSV

The direct download is a live dataset and may have changed since the guide snapshot. To reproduce the guide exactly, put the frozen CSV in My Drive/cta_bus_ridership.csv before running this cell. Otherwise, the cell downloads the current official export.

In [ ]:
from urllib.request import urlretrieve
import pandas as pd

REFERENCE_SHA256 = '6a999ea344084036310a779886bc6a72de6bad24cbb696e90822c83a337e608a'
DRIVE_CSV = Path('/content/drive/MyDrive/cta_bus_ridership.csv')
CSV_PATH = Path('/content/cta_bus_ridership.csv')
if DRIVE_CSV.exists():
    CSV_PATH = DRIVE_CSV
    print('Using Drive CSV:', CSV_PATH)
else:
    CSV_URL = 'https://data.cityofchicago.org/api/views/jyb9-n7fm/rows.csv?accessType=DOWNLOAD'
    try:
        urlretrieve(CSV_URL, CSV_PATH)
        print('Downloaded current official CTA export.')
    except Exception as error:
        print('Automatic download failed:', error)
        uploaded = files.upload()
        csv_names = [name for name in uploaded if name.lower().endswith('.csv')]
        if not csv_names:
            raise FileNotFoundError('Upload the CTA daily bus ridership CSV and rerun this cell.')
        CSV_PATH = Path('/content') / csv_names[0]

source_sha256 = hashlib.sha256(CSV_PATH.read_bytes()).hexdigest()
print('CSV:', CSV_PATH)
print('SHA-256:', source_sha256)
print('Matches mentor frozen export:', source_sha256 == REFERENCE_SHA256)

from src.data_loader import load_bus_ridership
ridership = load_bus_ridership(CSV_PATH)
print(f'{len(ridership):,} route-date rows | {ridership.route.nunique()} routes | '
      f'{ridership.date.min().date()} through {ridership.date.max().date()}')
print('Observed zero-ridership rows:', int(ridership.rides.eq(0).sum()))
if ridership.date.min() > pd.Timestamp('2024-08-01') or ridership.date.max() < pd.Timestamp('2026-07-31'):
    raise ValueError('This v3 comparison needs source history from 2024-08-01 through 2026-07-31.')

### Optional: download the validated CSV to your computer

Run the next cell after the data-loading cell finishes. Colab will transfer the exact validated file in `CSV_PATH` to your browser downloads; its SHA-256 was printed above.

In [ ]:
from google.colab import files
files.download(str(CSV_PATH))

## 2. Match the eight extracted GTFS snapshots

The imported schedules use the latest whole-network archive captured strictly before each target date. Same-day captures are excluded because their within-day availability is unknown. Extracted table fingerprints are reported; the original ZIP hash cannot be checked after extraction.

In [ ]:
import pandas as pd
from src.gtfs_service import build_service_table

GTFS_ROOT = Path('/content/drive/MyDrive/CTA_GTFS')
source_manifest = pd.read_csv(REPO / 'data/raw/gtfs_archive_manifest.csv', dtype='string').fillna('')
missing = []
for row in source_manifest.itertuples(index=False):
    folder = GTFS_ROOT / row.archive_id
    if not folder.is_dir():
        missing.append(str(folder))
if missing:
    raise FileNotFoundError('These extracted folders are missing from My Drive/CTA_GTFS: ' + ', '.join(missing))

manifest = source_manifest.copy()
manifest['zip_path'] = [str(GTFS_ROOT / archive_id) for archive_id in manifest['archive_id']]
manifest_path = Path('/content/gtfs_archive_manifest_extracted.csv')
manifest.to_csv(manifest_path, index=False)

OUTPUT_DIR = Path('/content/drive/MyDrive/CTA_RouteRanker_v3')
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
service_path = OUTPUT_DIR / 'cta_route_date_demand_service_v3.csv'
gtfs_report_path = OUTPUT_DIR / 'gtfs_service_import_v3.json'
gtfs_report = build_service_table(
    CSV_PATH, manifest_path, service_path, gtfs_report_path,
    snapshot_selection_policy='strictly_before_service_date',
)
print('Service rows:', gtfs_report['matched_route_date_rows'], 'matched route/date records')
print('Snapshot policy:', gtfs_report['snapshot_selection_policy'])
print('GTFS audit:', gtfs_report_path)

## 3. Compare baselines and models, then save the selected artifact

Demand-only validation/test uses the guide's larger partitions. To measure schedule value fairly, demand-only and service-aware candidates are compared on identical matched route/date rows. The selected overall paired configuration is chosen on validation MAE only. After test scoring, the selected estimator is refit through the latest available source date for the serving artifact; this refit does not change the reported test metrics. The test set was seen in earlier project iterations, so interpret test metrics as development evidence. The replay uses actual rides through D−1 for the next day's rolling prediction; this is not a single forecast issued months ahead.

In [ ]:
from src.train_v3 import run_v3_experiment

model_path = OUTPUT_DIR / 'cta_ridership_v3.joblib'
report_path = OUTPUT_DIR / 'model_selection_v3.json'
forecast_path = OUTPUT_DIR / 'cta_june_july_replay_v3.csv'
result = run_v3_experiment(
    ridership_path=CSV_PATH, service_path=service_path,
    model_path=model_path, report_path=report_path,
    replay_forecasts_path=forecast_path, reporting_delay_days=1,
    expected_reference_sha256=REFERENCE_SHA256,
)

print('Selected on validation:', result['model_selection'])
print('Demand-only reference test:', result['demand_only_reference_experiment']['selected_test_metrics'])
print('Paired matched test comparison:')
display(pd.DataFrame(result['paired_matched_experiment']['test_metrics_common_methods']).T.sort_values('mae'))
print('Paired matched coverage:', result['paired_matched_experiment']['coverage'])
print('Source fingerprint:', result['source'])
print('Saved model:', model_path)
print('Saved report:', report_path)
print('Forecasts (no actual target column):', forecast_path)

## 4. Download the result bundle

The ZIP contains the selected model bundle, model-selection report, GTFS import audit, forecast-only replay CSV, and a separate actuals file for offline scoring. Keep the source CSV and all reported hashes with your project record.

In [ ]:
import zipfile

bundle_path = OUTPUT_DIR / 'cta_route_ranker_v3_results.zip'
actuals_path = forecast_path.with_name(forecast_path.stem + '_actuals_for_scoring.csv')
with zipfile.ZipFile(bundle_path, 'w', compression=zipfile.ZIP_DEFLATED) as bundle:
    for artifact in [model_path, report_path, gtfs_report_path, forecast_path, actuals_path]:
        bundle.write(artifact, arcname=artifact.name)
print('Saved results in Drive:', bundle_path)
files.download(str(bundle_path))